# Reproducible benchmark scenario

This notebook explains the offline regression guard with checked-in fixtures. It does not run native benchmarks; it reads deterministic JSON inputs and the threshold table committed to the repository.

In [ ]:
from pathlib import Path
import json
import re

repo_root = Path.cwd()
base_path = repo_root / "benches" / "regression" / "sample-base.json"
current_path = repo_root / "benches" / "regression" / "sample-current.json"
thresholds_path = repo_root / "conductor" / "performance-thresholds.md"

base = json.loads(base_path.read_text(encoding="utf-8"))["benchmarks"]
current = json.loads(current_path.read_text(encoding="utf-8"))["benchmarks"]
thresholds_md = thresholds_path.read_text(encoding="utf-8")

assert base and current and thresholds_md
len(base), len(current)

## Parse the threshold table

The committed threshold table is the source of truth for accepted benchmark IDs and regression limits.

In [ ]:
thresholds = {}
for line in thresholds_md.splitlines():
    match = re.match(r"\| `([^`]+)` \| .*? \| (\d+)% \| .*? \| (blocking|advisory) \|", line)
    if match:
        benchmark_id, percent, gate = match.groups()
        thresholds[benchmark_id] = {"limit": int(percent) / 100, "gate": gate}

base_ids = {item["id"] for item in base}
current_ids = {item["id"] for item in current}

assert base_ids == current_ids == set(thresholds)
thresholds

## Compare fixture means

The scenario computes relative change as `(current_mean - base_mean) / base_mean`. Lower wall-clock means are better, so positive changes are regressions.

In [ ]:
base_by_id = {item["id"]: item["mean"] for item in base}
current_by_id = {item["id"]: item["mean"] for item in current}

comparison_rows = []
for benchmark_id in sorted(thresholds):
    base_mean = base_by_id[benchmark_id]
    current_mean = current_by_id[benchmark_id]
    change = (current_mean - base_mean) / base_mean
    limit = thresholds[benchmark_id]["limit"]
    comparison_rows.append(
        {
            "benchmark": benchmark_id,
            "change_percent": round(change * 100, 2),
            "limit_percent": round(limit * 100, 2),
            "gate": thresholds[benchmark_id]["gate"],
            "within_limit": change <= limit,
        }
    )

assert all(row["within_limit"] for row in comparison_rows)
comparison_rows

## Why this stays reproducible

The notebook uses only committed files, deterministic arithmetic, and assertions. Native benchmark collection remains owned by the benchmark harness, while this documentation asset teaches how the guard interprets already-collected results.